# Distributions, redundancy, and cradinality

In [ ]:
import sys, glob, warnings

warnings.filterwarnings("ignore")
sys.path.append("../../")  # lets us import helper.py
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from IPython.display import display
from helper import *

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 50)

DATA_FILE = glob.glob("../../data/melbourne/*.csv")[0]
TARGET = "Price"
df = pd.read_csv(DATA_FILE)


df["Date"] = pd.to_datetime(
    df["Date"], dayfirst=True, errors="coerce"
)  # sale date: text - datetime
df["Postcode"] = (
    df["Postcode"].astype("Int64").astype(str)
)  # a code, not a quantity - text
LAT = next(
    c for c in df.columns if c.lower().startswith("lat")
)  # the Kaggle file spells it "Lattitude"
LON = next(c for c in df.columns if c.lower().startswith("long"))

num_cols, cat_cols = split_columns(
    df, TARGET
)  # feature lists WITHOUT the target (Date is excluded from both)
print(
    df.shape,
    "| numeric features:",
    len(num_cols),
    "| categorical features:",
    len(cat_cols),
)


## 2a) Distributions

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 4))
sns.histplot(df[TARGET].dropna(), bins=60, kde=True, ax=ax[0])
ax[0].set_title(
    f"Price (skew {df[TARGET].skew():.2f}, kurtosis {df[TARGET].kurt():.2f})"
)
sns.histplot(np.log1p(df[TARGET].dropna()), bins=60, kde=True, ax=ax[1])
ax[1].set_title(
    f"log(Price) (skew {np.log1p(df[TARGET]).skew():.2f}) - diagnostic only"
)
plt.tight_layout()
plt.show()
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
stats.probplot(df[TARGET].dropna(), dist="norm", plot=ax[0])
ax[0].set_title("Q-Q: Price")
stats.probplot(np.log1p(df[TARGET].dropna()), dist="norm", plot=ax[1])
ax[1].set_title("Q-Q: log(Price)")
plt.show()

In [ ]:
hist_grid(df, num_cols)
display(skew_kurt_table(df, num_cols))

In [ ]:
cats = [c for c in cat_cols if df[c].nunique() <= 15]
fig, axes = plt.subplots(
    int(np.ceil(len(cats) / 2)), 2, figsize=(15, 4 * int(np.ceil(len(cats) / 2)))
)
for ax, col in zip(np.array(axes).ravel(), cats):
    sns.countplot(
        data=df, x=col, order=df[col].value_counts().index, ax=ax, color="steelblue"
    )
    ax.set_title(col)
    ax.tick_params(axis="x", rotation=40)
plt.tight_layout()
plt.show()

## 2b) Multicollinearity and redundancy

In [ ]:
plt.figure(figsize=(10, 8))
sns.heatmap(
    df[num_cols + [TARGET]].corr(),
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0,
    vmin=-1,
    vmax=1,
)
plt.title("Pearson correlation incl. Price")
plt.show()
display(high_corr_pairs(df, num_cols, 0.7))
display(vif_table(df, num_cols))

In [ ]:
display(pd.crosstab(df["Rooms"], df["Bedroom2"]))  # does Bedroom2 just repeat Rooms?
print(
    "Rows where Bedroom2 == Rooms:",
    round((df["Bedroom2"] == df["Rooms"]).mean() * 100, 1),
    "%",
)
# Propertycount should be constant inside a suburb (a suburb attribute, not a property attribute):
print(
    "Share of suburbs with exactly 1 distinct Propertycount:",
    round((df.groupby("Suburb")["Propertycount"].nunique() == 1).mean() * 100, 1),
    "%",
)
print(
    "Suburbs per Postcode / Postcodes per suburb (location redundancy):",
    df.groupby("Postcode")["Suburb"].nunique().mean().round(2),
    "/",
    df.groupby("Suburb")["Postcode"].nunique().mean().round(2),
)

## 2c) High credinality

In [ ]:
card = cardinality_table(df, cat_cols)
display(card)
fig, ax = plt.subplots(1, 2, figsize=(15, 4.5))
card["unique_%_of_rows"].sort_values().plot.barh(ax=ax[0], color="steelblue")
ax[0].set_title("Unique values as % of rows")
sns.histplot(df["Suburb"].value_counts().values, bins=40, ax=ax[1])
ax[1].set_title("Rows per suburb (long tail?)")
plt.tight_layout()
plt.show()